<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/omp_removal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***OMP removal calculation***


In [ ]:
import math as mh
import pandas as pd

def omp_removal(lanes,Partable2,dailyflow,Treatment):
    # OMP removal design
    # This code calculates the dimensions of OMP removal stage in a water treatment plant.
    #treatmenttechnologies2 = treatmenttechnologies.astype(str)

    # Input parameters
    print("")
    print("OMP REMOVAL PROCESS")
    flowRate = round(dailyflow/24,1) # m3/h
    print(f"The daily flow rate is {flowRate}m3/h")
    # Calculate surface loading rate
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "granular activated carbon filtration (gac)").any():
      ompArea, Eomp, Homp,ompdepth=gac(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "powdered activated carbon dosing (pac)").any():
      ompArea, Eomp, Homp,ompdepth=pac(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "mp-uv/h2o2").any():
      ompArea, Eomp, Homp,ompdepth=mpuvh2o2(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "nf/ro").any():
      ompArea, Eomp, Homp,ompdepth=nfro(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "ozonation").any():
      ompArea, Eomp, Homp,ompdepth=ozon(flowRate,lanes) #, units    else:
    else:
      ompArea, Eomp, Homp,ompdepth=0,0,0

    return ompArea, Eomp, Homp,ompdepth

def gac(flowRate,lanes):
# Set the number of units per lane
    while True:
        units_str = input("Number of different GAC units per lane:")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number greater than or equal to 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")
    flowunit = flowRate / (units*lanes)
    #Set Empty Bed Contact time (EBCT)
    while True:
        EBCT = int(input("Empty Bed contact time (min) (min 10min -  max 20min default 15min ):") or 15)
        if 10 <= EBCT <= 20:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    EBCT= EBCT/60 #convert minutes in hours
    Vgacunit= EBCT*flowunit/60 #convert minutes in hours
    qmax=100 #maximum adsorption 100mg/kg
    while True:
        HLR = int(input("The hydraulic loading rate (HLR) (m³/m²·hr, min 5 - max 15 - default 10): ") or 10)
        if 5 <= HLR <= 15:
            break
        else:
            print("Invalid input. Enter a value between 0.3 and 8.")
    ompunit=mh.ceil(flowunit/HLR) # cross-sectional area (m2)
    print(f"The GAC unit section area is {ompunit}m2")
    ompArea=mh.ceil(ompunit*units*lanes) #m2
    print(f"The overall area of the GAC tanks is {ompArea}m2",)
    W = Vgacunit/ompunit
    #print(f"The bed height at each GAC unite is {W}m3")
    while True:
        dgac = float(input("Carbon size in mm (min 0.4mm -  max 2.5mm default 2mm ):") or 2)
        if 0.4 <= dgac <= 2.5:
            break
        else:
            print("Invalid input. Enter a value between 0.4 and 2.5.")
    Homp=1/(dgac/1000) #PA
    Homp=round(0.102*Homp/1000,1) #Pa to mH20
    Eomp=0
    print(f"The hydraulic losses in the GAC are {Homp}m")
    ompdepth=W
    return ompArea, Eomp, Homp,ompdepth

def pac(flowRate,lanes):

    flowlane=flowRate/lanes # m3/h
    remrate=int(input("Set the removal rate (%)"))
    if remrate>=80:
       PACdos=float(input("Set the PAC concentration (minimum 25 mg/l):"))
    else:
       PACdos=float(input("Set the PAC concentration (maximum 25 mg/l):"))
    Massgacunit=flowRate*PACdos*24/1000 #kg/day
    print(f"The daily PAC requirements are {Massgacunit}kg/day")
    Pacsol=500 #assuming %500kg/m3 solution PAC
    PACvolday=(Massgacunit/Pacsol) #m3/day
    while True:
        ndays = int(input("Set the number of PAC storage days (min 1day - max 40days - default 20days):") or 2)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    PACvol=PACvolday*ndays #m3
    PACarea=mh.ceil(PACvol/2.5) # assuming 2.5 high storage tanks
    print(f"The are requirement for the PAC storage for {ndays} days is {PACarea}m2")
    PACcontatvolumeunit=mh.ceil(1.1*flowlane*0.5) # one per lane
    PACcontatareaunit=mh.ceil(PACcontatvolumeunit/2.5)
    PACcontatarea=mh.ceil(PACcontatareaunit*lanes)
    print(f"The overall area requirement for the PAC contact basin is {PACcontatarea}m2")
    ompArea=PACcontatarea+PACarea
    print(f"The overall area requirement for the PAC process is {ompArea}m2")
    PACpumpflow=PACvolday*1000/24 #l/h
    print(f"The PAC pump flow rate is {PACpumpflow}l/h")
    Pomp=PACpumpflow*1000*9.81*10/(3600*1000)
    Eomp=mh.ceil(Pomp*24)
    print(f"The daily energy comsuption of the PAC dosing pump is {Eomp}KWh")
    Homp=0.4
    ompdepth=0
    return ompArea, Eomp, Homp,ompdepth

def mpuvh2o2(flowRate, lanes):
    """
    Calculate parameters for MP-UV/H2O2 advanced oxidation process.

    Parameters:
    - flowRate (float): total flow rate in m³/h
    - lanes (int): number of UV lanes

    Returns:
    - ompArea (float): total area of the UV system (m²)
    - H2O2_monthly (float): H2O2 requirement per month (kg)
    - total_power (float): total power consumption (kW)
    """

    # Flow per UV lane
    flowlane = flowRate / lanes  # m³/h

    # Ask for UV dose (600–1000 mJ/cm²)
    UVdose = int(input("Set the UV dose (between 600-1000 mJ/cm²): "))
    if UVdose < 600 or UVdose > 1000:
        print("Warning: UV dose should be between 600 and 1000 mJ/cm²")

    # Ask user to select water quality
    print("\nSelect water quality:")
    print("1. Low organic content (e.g., drinking water, groundwater)")
    print("2. Moderate organic content (e.g., wastewater, pharmaceuticals)")
    print("3. High organic content (e.g., landfill leachate, industrial waste)")
    water_quality_choice = int(input("Enter choice (1-3): "))

    # Adjust H₂O₂ dosage based on UV dose and water quality
    if water_quality_choice == 1:
        H2O2_dose_mg_per_l = max(2, 10 - UVdose / 150)  # Low organics (2–5 mg/L typical)
        water_quality = "low"
    elif water_quality_choice == 2:
        H2O2_dose_mg_per_l = max(5, 15 - UVdose / 100)  # Moderate organics (5–15 mg/L typical)
        water_quality = "moderate"
    elif water_quality_choice == 3:
        H2O2_dose_mg_per_l = max(10, 25 - UVdose / 80)  # High organics (10–50 mg/L typical)
        water_quality = "high"
    else:
        H2O2_dose_mg_per_l = 7  # Default if invalid input
        water_quality = "moderate (default)"

    print(f"\nSelected water quality: {water_quality}")
    print(f"Suggested H₂O₂ dosage: {H2O2_dose_mg_per_l:.1f} mg/L")

    while True:
        retention_time = int(input("Set the retention time in minutes(min 3min - max 10min - default 5): ") or 5)
        if 3 <= retention_time <= 10:
            break
        else:
            print("Invalid input. Enter a value between 3 and 10.")

    # Calculate UV tank volume per lane (m³)
    UV_tank_volume = flowlane * retention_time / 60

    # Ask for tank height
    depth = float(input("Desired depth of the MP-UV tank (m): "))

    # Surface area per lane (m²)
    area_per_lane = UV_tank_volume / depth

    # Total area for all lanes
    ompArea = round(area_per_lane * lanes, 2)
    print(f"The total area of the MP-UV process (m²) is {ompArea}")

    # Power requirement per lane (kW)
    power_per_lane = 30 * flowlane / 1000
    total_UV_power = power_per_lane * lanes


    # Monthly H2O2 requirement (kg)
    H2O2_daily =round(H2O2_dose_mg_per_l * 0.001 * flowRate * 24,2)
    H2O2_monthly = H2O2_daily * 30
    print(f"H₂O₂ requirements per day is {H2O2_daily} kg")
    print(f"H₂O₂ requirements per month is {H2O2_monthly}kg")

    # Power required for H2O2 dosage pump (assumed flow = 500,000 L/h, pressure = 0.5 bar)
    H2O2_pump_power = (0.000278 * 500000 / 0.5) / 1000  # kW
    print("Power consumption for the H₂O₂ dosage pump (kW):", round(H2O2_pump_power, 2))

    # Total power consumption
    total_power = round(total_UV_power + H2O2_pump_power, 2)
    print("Total power consumption for the MP-UV/H₂O₂ system (kW):", total_power)

    Eomp=round(total_power*24)
    print(f"The daily energy comsuption of the MP-UV/H2O2 system is {Eomp}KWh")
    Homp=0.4
    ompdepth=depth
    return ompArea, Eomp, Homp, ompdepth

def nfro(flowRate,lanes):
    """
    Calculates NF/RO system design parameters.

    Args:
        flow_Rate: Flow rate in m3/day
        flowlane: Flow rate per lane in m3/day
        flowunit: Flow rate per unit in m3/day
        delta_OMP:  MP removal in mg/L
        ini_OMP: Initial hardness in mg/L
        nf_rejection: NF membrane rejection
        ro_rejection: RO membrane rejection
        nf_permeate_per_membrane: NF permeate flow area per element in m3/day
        ro_permeate_per_membrane: RO permeate flow area per element in m3/day
        operating_pressure_nf: Operating pressure for NF stage in bar
        operating_pressure_ro: Operating pressure for RO stage in bar
        pump_efficiency: Pump efficiency (decimal)
        membrane_orientation: 'vertical' or 'horizontal'
        membranes_per_vessel: Membranes per vessel (set 6 - 8 per vessel)

    Returns:
        A dictionary containing:
            - nf_membranes: Number of NF membranes
            - ro_membranes: Number of RO membranes
            - nf_membrane_area: Total NF membrane area
            - ro_membrane_area: Total RO membrane area
            - nf_permeate_flow: NF permeate flow rate
            - ro_permeate_flow: RO permeate flow rate
            - energy_consumption_nf: Energy consumption for NF stage (kWh/day)
            - energy_consumption_ro: Energy consumption for RO stage (kWh/day)
            - concentration_polarization_factor: Estimated concentration polarization factor (simplified)
            - scaling_potential: Simplified scaling potential indicator (higher values indicate higher risk)
    """

    units=float(input("Enter the number of units per lane(minimum 2 units): "))
    while True:
        try:
            if units >= 1:
                print("The total number of NF/RO units in the DWTP is",units*lanes)
                break
            else:
                print("One unit per lane is required")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Nominal flow
    tbr=1
    flowlane=flowRate/lanes # m3/h
    flowunit=flowRate/(lanes*(units-1)) # m3/h
    flow_rate=flowunit*24 #m3/day

    # OMP removal
    in_OMP=float(input("Set initial OMP:"))
    target_OMP = float(input("set target OMP:"))
    delta_OMP = in_OMP - target_OMP # OMP removal
    Cout_nf = in_OMP - in_OMP*0.8 # assuming 80% removal of OMPs in NF
    nf_rejection=1-(Cout_nf/in_OMP)
    rf_rejection=1-(target_OMP/Cout_nf)

    # Calculate permeate flow rates
    nf_recovery=float(input("Set the NF recovery rate (estimated between 0.7-0.9):"))
    ro_recovery=float(input("Set the RO rejection rate (estimated between 0.6-0.95):"))
    nf_permeate_flow = flow_rate * nf_recovery
    ro_permeate_flow = nf_permeate_flow * ro_recovery
    print("The NF permeate flow rate (m3/day) is",nf_permeate_flow)
    print("The RO permeate flow rate (m3/day) is",ro_permeate_flow)
    con_flow=flow_rate-ro_permeate_flow

    # Calculate number of membranes
    print("The dimensions of the membranes is set using commercial membranes specifications of two main commercial brands Nitto and CSM")
    print("The selected nanofiltration (NF) membrane has a length A of 1.1 meters, a width of 0.2 meters and feed pipe of 30 mm")
    Jc_nf=int(input("The Flux Rate of the NF system is (15-30 L/m2*h):"))
    print("The selected reverse osmosis (RO) membrane has a length A of 1.1 meters, a width of 0.2 meters and feed pipe of 20 mm")
    Jc_ro=int(input("The Flux Rate of the RO system is (10-20 L/m2*h):"))
    print("The active area per membrane is 37.2m2 for NF membranes and 40.9 per RO membranes")
    nf_area_per_membrane=37.2
    ro_area_per_membrane=40.9
    nf_area_oveall=nf_permeate_flow*41.7/Jc_nf
    ro_area_oveall=ro_permeate_flow*41.7/Jc_ro
    nf_membranes = mh.ceil(nf_area_oveall / nf_area_per_membrane)
    ro_membranes = mh.ceil(ro_area_oveall / ro_area_per_membrane)

    # Calculate energy consumption (kWh/day)
    # 1 bar = 10.2 mH20
    print("The operating pressure of the NF membranes is 6 bar")
    print("The operating pressure of the RO membranes is 8 bar")
    operating_pressure_nf=6
    operating_pressure_ro=8
    pump_efficiency=0.75
    nf_power = (nf_permeate_flow *9.81) * (operating_pressure_nf * 10.2) / (86400 * pump_efficiency)  # kW
    ro_power = (ro_permeate_flow *9.81) * (operating_pressure_ro * 10.2) / (86400 * pump_efficiency)  # kW
    Punit=nf_power+ro_power
    Pomp=Punit*lanes*(units-1)
    energy_consumption_nf = round(nf_power * 24)  # kWh/day
    energy_consumption_ro = round(ro_power * 24)  # kWh/day
    print("The energy consumption (KWh) for the NF stage is",energy_consumption_nf)
    print("The energy consumption (KWh) for the RO stage is",energy_consumption_ro)
    ompenergyunit=round(energy_consumption_nf+energy_consumption_ro)
    print("The total energy consumption per unit per day (kWh) is",ompenergyunit)
    Eomp=round(ompenergyunit*(units-1)*lanes)
    print("The total energy consumption per day (kWh) is",ompenergy)

    # Estimate concentration polarization factor (simplified) - This requires more detailed modeling
    membrane_orientation=input("Enter the membrane orientation (vertical or horizontal):")
    if membrane_orientation == 'vertical':
        concentration_polarization_factor = 1.2  # Example value, adjust based on actual flow conditions
    elif membrane_orientation == 'horizontal':
        concentration_polarization_factor = 1.5  # Example value, adjust based on actual flow conditions
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")

    # Simplified scaling potential indicator (higher values indicate higher risk)
    scaling_potential = (in_OMP - target_OMP) * concentration_polarization_factor

    #Set up the membranes per vessel
    membranes_per_vessel=int(input("Set the number of membranes per vessel (6-8):"))
    nf_no_vesels=mh.ceil(nf_membranes/membranes_per_vessel)
    ro_no_vesels=mh.ceil(ro_membranes/membranes_per_vessel)

    # Membrane characteristics
    print("NF Membranes:", nf_membranes)
    print("RO Membranes:", ro_membranes)
    print("NF Membrane Area:", nf_area_per_membrane, "m2")
    print("RO Membrane Area:", ro_area_per_membrane, "m2")
    print("NF Permeate Flow:", nf_permeate_flow, "m3/day")
    print("RO Permeate Flow:", ro_permeate_flow, "m3/day")
    print("NF Energy Consumption:", energy_consumption_nf, "kWh/day")
    print("RO Energy Consumption:", energy_consumption_ro, "kWh/day")
    print("Concentration Polarization Factor:", concentration_polarization_factor)
    print("Scaling Potential:", scaling_potential)
    print("Number of vessels in the NF process:", nf_no_vesels)
    print("Number of vessels in the RO process:", ro_no_vesels)

    # OMP volume
    if membrane_orientation == 'vertical':
       nf_no_vesels_area=1.1*(0.3*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))*nf_no_vesels
       ro_no_vesels_area=1.1*(0.3*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))*ro_no_vesels
       ompunit=nf_no_vesels_area+ro_no_vesels_area
       pump_unit=round(1.2*ompunit)
       print("pump area:",pump_unit)
       ompArea=round((ompunit+pump_unit)*units*lanes)
       L=ompunit/2
       print("The total area of the NF/RO process (m2) is:",softarea)
    elif membrane_orientation == 'horizontal':
       nf_no_vesels_area=1.1*(1.2*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))
       ro_no_vesels_area=1.1*(1.2*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))
       ompunit=nf_no_vesels_area+ro_no_vesels_area
       pump_unit=round(1.2*ompunit)
       print("pump area:",pump_unit)
       ompArea=round((ompunit+pump_unit)*units*lanes)
       L=ompunit/2
       print("The total area of the NF/RO process (m2) is:",ompArea)
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")
    Dpm=0.3
    Homp=round(L*0.01+Dpm,2)
    ompdepth=L
    return ompArea, Eomp, Homp,ompdepth

def ozon(flowRate,lanes):
    while True:
        tbr = input("Set the required retention time (min 8min, max 20min, default 15):") or "15"
        try:
            tbr = int(tbr)
            if 8 <= tbr <= 20:
                break
            else:
                print("Invalid input. Enter a number between 8 and 20.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    while True:
        units_str = input("Number of different ozonation units per lane:")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number greater than or equal to 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    flowunit=flowRate/(lanes*units) # m3/h
    print('The Ozone dosage for disinfection only is assumed to be 2 mg/l')
    O3=2 #mg/l
    ozonevolume=flowunit*tbr/60
    D=float(input("Set the depth of the ozonation tank (m)"))
    ompunit=mh.ceil(ozonevolume/ozonedepth) #m2
    omparea=mh.ceil(disunitarea*lanes*units)
    print(f"The area of the ozonation unit is {ompunit}m2")
    print(f"The total area of the ozonation process tanks is {omparea}m2")
    ozonedem=O3*flowRate*0.001 #kg/h
    ozonedem=round(ozonedem*1.1,2) #safety factor
    print(f"The ozone demand is {ozonedem}kg/h")
    ozgensize=mh.ceil(ozonedem*10) #10m2 per kg produced
    print("The area required for the the ozone generators (m2):",ozgensize)
    omparea=mh.ceil(disarea+ozgensize)
    print("The total area of the ozonation process (ozonation tank and storage of ozonators) (m2):",omparea)
    eo3=15 # 15 kWh/kgO3
    Eomp=mh.ceil(eo3*ozonedem*24) #KW
    print(f"The daily energy consumption per hour required for the ozonation is {Eomp}KWh")
    L=3*mh.sqrt(disarea/3)
    Homp=round(L*0.01,2)
    ompdepth=D
    return ompArea, Eomp, Homp,ompdepth